# Layer 2 — 正常訂單瓶頸拆解

排除 Layer 1a + 1b 異常訂單後，分析正常訂單的瓶頸在哪。
將 total_duration 拆成四段：queue, db, device, inner_processing。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)
PARALLELISM = 4

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')

# Load anomaly flags
sys_flags = pd.read_csv('../data/system_anomaly_flags.csv')
usr_flags = pd.read_csv('../data/user_anomaly_flags.csv')
df = df.merge(sys_flags, on='order_id')
df = df.merge(usr_flags, on='order_id')

# Filter to normal orders only
df['is_anomaly'] = df['is_system_anomaly'] | df['is_user_anomaly']
normal = df[~df['is_anomaly']].copy()
print(f"Total: {len(df)}, Anomalies excluded: {df['is_anomaly'].sum()}, Normal: {len(normal)}")


Total: 30000, Anomalies excluded: 1517, Normal: 28483


## 1. Phase Duration 估算

每個 order 有 4 threads 並行處理，order-level 耗時 ≈ avg × file_count / 4

In [2]:
# Estimate order-level phase durations
normal['est_queue'] = normal['queue_duration_seconds']
normal['est_db'] = normal['db_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_device'] = normal['device_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_inner'] = normal['inner_processing_duration_avg_seconds'] * normal['file_count'] / PARALLELISM
normal['est_total'] = normal['est_queue'] + normal['est_db'] + normal['est_device'] + normal['est_inner']

# File count groups
bins = [0, 50, 300, 1000, 2000, 5000]
labels = ['<50', '50-300', '300-1000', '1000-2000', '2000+']
normal['fc_group'] = pd.cut(normal['file_count'], bins=bins, labels=labels, right=True)

print("Orders per file_count group:")
print(normal['fc_group'].value_counts().sort_index())


Orders per file_count group:
fc_group
<50          8626
50-300       8471
300-1000     7147
1000-2000    3111
2000+        1128
Name: count, dtype: int64


## 2. Model Validation

比較 est_total 與 actual total_duration，確認 phase decomposition 模型的準確度。

In [3]:
# Model validation: est_total vs actual
normal['ratio'] = normal['est_total'] / normal['total_duration_seconds']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: scatter est vs actual
ax = axes[0]
ax.scatter(normal['total_duration_seconds'], normal['est_total'], alpha=0.05, s=5, c='steelblue')
max_val = max(normal['total_duration_seconds'].quantile(0.99), normal['est_total'].quantile(0.99))
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect fit')
ax.set_title('Model Validation: Estimated vs Actual Duration')
ax.set_xlabel('Actual Total Duration (seconds)')
ax.set_ylabel('Estimated Total Duration (seconds)')
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
ax.legend()

# R² score
ss_res = ((normal['est_total'] - normal['total_duration_seconds']) ** 2).sum()
ss_tot = ((normal['total_duration_seconds'] - normal['total_duration_seconds'].mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
ax.text(0.05, 0.9, f'R² = {r2:.3f}', transform=ax.transAxes, fontsize=12)

# Right: ratio distribution by fc_group
ratio_by_group = normal.groupby('fc_group', observed=True)['ratio'].agg(['median', 'mean', 'count'])
ratio_by_group['median'].plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8)
axes[1].axhline(y=1.0, color='red', linestyle='--', label='Perfect = 1.0')
axes[1].set_title('Model Fit Ratio (est/actual) by File Count Group')
axes[1].set_xlabel('File Count Group')
axes[1].set_ylabel('Median Ratio (est_total / actual)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()
for i, (idx, row) in enumerate(ratio_by_group.iterrows()):
    axes[1].text(i, row['median'] + 0.02, f'{row["median"]:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_model_validation.png', dpi=150)
plt.close()
print(f"Saved: 2_model_validation.png")
print(f"\nModel R²: {r2:.3f}")
print(f"Ratio by group:")
print(ratio_by_group.to_string())
print(f"\n⚠️  file_count <50 組 ratio 偏低，模型低估約 {(1-ratio_by_group.loc['<50','median'])*100:.0f}%，"
      f"原因為固定開銷 (connection setup, scheduling) 在小訂單中佔比高。")


Saved: 2_model_validation.png

Model R²: 0.999
Ratio by group:
             median      mean  count
fc_group                            
<50        0.663363  0.696058   8626
50-300     1.128968  1.177457   8471
300-1000   1.034065  1.042698   7147
1000-2000  1.014224  1.016472   3111
2000+      1.006248  1.007239   1128

⚠️  file_count <50 組 ratio 偏低，模型低估約 34%，原因為固定開銷 (connection setup, scheduling) 在小訂單中佔比高。


## 2b. Overhead Estimation

對 `<50` 組，模型 `est_total = queue + (db+device+inner)*fc/4` 系統性低估，
原因是未包含固定開銷。用線性迴歸 `total = overhead + slope × file_count` 估算。

In [4]:
# Estimate fixed overhead via linear regression on small orders
from numpy.polynomial.polynomial import polyfit
small = normal[normal['file_count'] <= 50].copy()
coeffs = np.polyfit(small['file_count'], small['total_duration_seconds'], 1)
slope, intercept = coeffs[0], coeffs[1]
print(f"Linear regression (file_count ≤ 50):")
print(f"  total ≈ {intercept:.1f} + {slope:.2f} × file_count")
print(f"  Estimated fixed overhead: {intercept:.1f}s ({intercept/60:.1f} min)")
print(f"  Per-file marginal cost: {slope:.2f}s")
print(f"\n  This overhead includes: connection setup, queue processing, result aggregation, scheduling.")
print(f"  在 file_count <50 時，fixed overhead (~{intercept:.0f}s) 佔 total 的主要部分，")
print(f"  導致 phase decomposition model (est_total) 系統性低估 (median ratio = {ratio_by_group.loc['<50','median']:.2f})。")
print(f"  file_count ≥50 後，per-file processing 開始主導，model fit 回復正常 (ratio ≈ 1.0)。")
print(f"  ⇒ Phase breakdown 對 file_count <50 的結論需謹慎解讀。")


Linear regression (file_count ≤ 50):
  total ≈ 73.0 + 0.10 × file_count
  Estimated fixed overhead: 73.0s (1.2 min)
  Per-file marginal cost: 0.10s

  This overhead includes: connection setup, queue processing, result aggregation, scheduling.
  在 file_count <50 時，fixed overhead (~73s) 佔 total 的主要部分，
  導致 phase decomposition model (est_total) 系統性低估 (median ratio = 0.66)。
  file_count ≥50 後，per-file processing 開始主導，model fit 回復正常 (ratio ≈ 1.0)。
  ⇒ Phase breakdown 對 file_count <50 的結論需謹慎解讀。


## 3. Phase 佔比分析

In [5]:
# Compute phase proportions per group
phase_cols = ['est_queue', 'est_db', 'est_device', 'est_inner']
phase_labels = ['Queue', 'DB', 'Device', 'Inner Processing']

group_means = normal.groupby('fc_group', observed=True)[phase_cols].mean()
group_means.columns = phase_labels

# Proportions
group_pct = group_means.div(group_means.sum(axis=1), axis=0) * 100

print("Phase proportion by file_count group (%):")
print(group_pct.round(1).to_string())
print()
print("Phase absolute means (seconds):")
print(group_means.round(1).to_string())


Phase proportion by file_count group (%):
           Queue    DB  Device  Inner Processing
fc_group                                        
<50         26.7  19.4    38.9              14.9
50-300       5.4  25.1    50.5              19.0
300-1000     1.5  26.2    52.5              19.9
1000-2000    0.6  26.4    53.0              20.0
2000+        0.3  25.9    54.2              19.7

Phase absolute means (seconds):
           Queue      DB  Device  Inner Processing
fc_group                                          
<50         14.4    10.5    21.0               8.0
50-300      14.3    66.9   134.6              50.5
300-1000    14.3   248.0   497.1             188.2
1000-2000   14.0   572.1  1149.4             432.7
2000+       14.4  1336.0  2792.9            1013.8


## 4. 圖表

In [6]:
# Chart 1: Phase proportion stacked bar (percentage) + absolute
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#f39c12', '#3498db', '#e74c3c', '#2ecc71']

group_pct.plot(kind='bar', stacked=True, ax=axes[0], color=colors)
axes[0].set_title('Phase Proportion by File Count Group (%)')
axes[0].set_xlabel('File Count Group')
axes[0].set_ylabel('Proportion (%)')
axes[0].legend(title='Phase', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

group_means.plot(kind='bar', stacked=True, ax=axes[1], color=colors)
axes[1].set_title('Phase Duration by File Count Group (seconds)')
axes[1].set_xlabel('File Count Group')
axes[1].set_ylabel('Mean Duration (seconds)')
axes[1].legend(title='Phase', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_phase_breakdown_bars.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 2_phase_breakdown_bars.png")


Saved: 2_phase_breakdown_bars.png


In [7]:
# Chart 2: Duration vs file_count scatter with trend
fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(normal['file_count'], normal['total_duration_seconds'], alpha=0.1, s=5, c='steelblue')

z = np.polyfit(normal['file_count'], normal['total_duration_seconds'], 1)
p = np.poly1d(z)
x_line = np.linspace(normal['file_count'].min(), normal['file_count'].max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend: {z[0]:.2f}x + {z[1]:.0f}')

ax.set_title('Total Duration vs File Count (Normal Orders)')
ax.set_xlabel('File Count')
ax.set_ylabel('Total Duration (seconds)')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_duration_vs_filecount.png', dpi=150)
plt.close()
print("Saved: 2_duration_vs_filecount.png")


Saved: 2_duration_vs_filecount.png


In [8]:
# Chart 3: Percentile analysis per group
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
phase_map = {'est_queue': 'Queue', 'est_db': 'DB', 'est_device': 'Device', 'est_inner': 'Inner Processing'}

for idx, (col, label) in enumerate(phase_map.items()):
    ax = axes[idx // 2][idx % 2]
    percentiles = normal.groupby('fc_group', observed=True)[col].quantile([0.5, 0.95, 0.99]).unstack()
    percentiles.columns = ['P50', 'P95', 'P99']
    percentiles.plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
    ax.set_title(f'{label} Duration Percentiles by File Count Group')
    ax.set_xlabel('File Count Group')
    ax.set_ylabel('Duration (seconds)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / '2_phase_percentiles.png', dpi=150)
plt.close()
print("Saved: 2_phase_percentiles.png")


Saved: 2_phase_percentiles.png


## 5. Summary

In [9]:
biggest = group_pct.idxmax(axis=1)
print("Biggest bottleneck per file_count group:")
for g, phase in biggest.items():
    print(f"  {g}: {phase} ({group_pct.loc[g, phase]:.1f}%)")

print(f"\n=== Layer 2 Summary ===")
print(f"Normal orders analyzed: {len(normal)}")
print(f"Overall dominant phase: {group_pct.mean().idxmax()} ({group_pct.mean().max():.1f}% avg)")
print(f"Model R²: {r2:.3f}")


Biggest bottleneck per file_count group:
  <50: Device (38.9%)
  50-300: Device (50.5%)
  300-1000: Device (52.5%)
  1000-2000: Device (53.0%)
  2000+: Device (54.2%)

=== Layer 2 Summary ===
Normal orders analyzed: 28483
Overall dominant phase: Device (49.8% avg)
Model R²: 0.999
